In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes
print("\n\n インストール成功！！！！")

# 2026/10/02時点では動作したが、依存関係が謎すぎて動作の保証ができない。
# requirements.txtに出力はしておくが、google colabなどの仕様変更があれば動作しない可能性あり。

In [ ]:

# !pip install --no-deps gcsfs==2025.12.0

# Unslothのインストール自体は成功するが、gcsfsにエラーが出るので、ピンポイントで修正する場合はこのセルを実行。
# ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.
# This behaviour is the source of the following dependency conflicts.gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.


In [ ]:
from unsloth import FastLanguageModel, get_chat_template # WARNINGが出たので、最初にimportする。


import json
import os
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
seed = 42

max_seq_length = 2048 # 最大トークン数
dtype = None # 自動検出
load_in_4bit = True # 4bit量子化（QLoRA）でメモリを節約

## 1. モデル設定

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
#  LoRA（PEFT）の設定
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRAのランク（8〜64が推奨）
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,     # 0に設定すると最適化されて高速化
    bias = "none",
    use_gradient_checkpointing = "unsloth", # メモリ節約
    random_state = seed,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# チャットテンプレートを適用する関数
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5", # Qwen2.5の公式フォーマットを指定
)

In [ ]:
# プロンプトのフォーマットする
def formatting_prompts_func(examples):

    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]

    return { "text" : texts }

## 2. データセットの準備

In [ ]:
DATA_DIR = "./data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

In [ ]:
file_name = DATA_DIR + "/lora_html.jsonl"

dataset = load_dataset("json", data_files=file_name, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

## 3. 学習

In [ ]:
OUTPUT_DIR = "./output"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [ ]:
# 学習関数の設定

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # 短いテキストが多いためパックしない
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # セリフの応答集が約50件なので、少なめに設定
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = seed,
        output_dir = OUTPUT_DIR,
    ),
)

In [ ]:
# 学習スタート
trainer_stats = trainer.train()

## 4. 動作確認

In [ ]:
# 推論モードに切り替え
FastLanguageModel.for_inference(model)

In [ ]:
messages = [
    {"role": "system", "content": "あなたは冷徹で厳格、無駄を嫌い、プログラマ（ユーザー）に対して非常に厳しい態度を取るC言語の擬人化キャラクター「C言語くん」です。"},
    {"role": "user", "content": "LLMのLoRAを行うためのunslothライブラリの挙動がわけわかめ"}
]

In [ ]:
# チャット形式のインプットを作成
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 回答の生成を促す
    return_tensors = "pt",
).to("cuda")

In [ ]:
# LLMからの返事を表示
rate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.decode(outputs = model.geneutputs[0][len(inputs[0]):], skip_special_tokens=True))

## 5. LoRAファイルの保存

In [ ]:

model.save_pretrained_merged(
    "html_chan_lora",
    tokenizer,
    save_method = "lora" # LoRAの差分ファイルの保存。ただし、ベースモデルがどれかの情報も含む。
  )

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("html_chan_lora", "zip", "html_chan_lora")
files.download("html_chan_lora.zip")